# Phase 2 — Data Cleaning + Taiwan Dependency Index

**Project:** Assessing Thailand's Exposure to Taiwan Semiconductor Supply Chain Disruption

**Dataset:** UN Comtrade, Thailand imports, HS 8542 Electronic integrated circuits, 2014–2024.

**Important methodology note:** `Other Asia, nes` is used as a proxy for Taiwan/Chinese Taipei because Taiwan is not separately reported in this dataset. This should be stated as a limitation in the report.


In [ ]:
import csv
from pathlib import Path
import plotly.graph_objects as go

RAW_PATH = Path('../data/raw/TradeData.csv')
if not RAW_PATH.exists():
    RAW_PATH = Path('../TradeData.csv')

rows = []
with open(RAW_PATH, encoding='latin1', newline='') as f:
    reader = csv.DictReader(f)
    for row in reader:
        row.pop(None, None)
        rows.append(row)

print('Rows:', len(rows))
print('Years:', sorted({int(r['refYear']) for r in rows}))
print('Reporter:', sorted({r['reporterDesc'] for r in rows}))
print('Flow:', sorted({r['flowDesc'] for r in rows}))
print('Commodity:', sorted({(r['cmdCode'], r['cmdDesc']) for r in rows}))


## 1. Keep useful columns only


In [ ]:
clean_rows = []
for r in rows:
    clean_rows.append({
        'year': int(r['refYear']),
        'partner_country': r['partnerDesc'],
        'trade_value_usd': float(r['primaryValue'] or 0),
        'reporter': r['reporterDesc'],
        'flow': r['flowDesc'],
        'hs_code': r['cmdCode'],
        'commodity': r['cmdDesc'],
    })

clean_rows = sorted(clean_rows, key=lambda x: (x['year'], -x['trade_value_usd'], x['partner_country']))
clean_rows[:5]


## 2. Calculate total imports and Other Asia, nes imports by year


In [ ]:
years = sorted({r['year'] for r in clean_rows})
world_by_year = {}
taiwan_proxy_by_year = {}
country_by_year = {}

for r in clean_rows:
    key = (r['year'], r['partner_country'])
    country_by_year[key] = country_by_year.get(key, 0) + r['trade_value_usd']
    if r['partner_country'] == 'World':
        world_by_year[r['year']] = world_by_year.get(r['year'], 0) + r['trade_value_usd']
    if r['partner_country'] == 'Other Asia, nes':
        taiwan_proxy_by_year[r['year']] = taiwan_proxy_by_year.get(r['year'], 0) + r['trade_value_usd']

dependency_rows = []
prev_total = None
prev_taiwan = None

for y in years:
    total = world_by_year.get(y, 0)
    taiwan_proxy = taiwan_proxy_by_year.get(y, 0)
    dependency_rows.append({
        'year': y,
        'total_hs8542_import_usd': total,
        'other_asia_nes_taiwan_proxy_import_usd': taiwan_proxy,
        'taiwan_proxy_dependency_pct': taiwan_proxy / total * 100 if total else None,
        'total_import_yoy_growth_pct': (total / prev_total - 1) * 100 if prev_total else None,
        'taiwan_proxy_yoy_growth_pct': (taiwan_proxy / prev_taiwan - 1) * 100 if prev_taiwan else None,
    })
    prev_total = total
    prev_taiwan = taiwan_proxy

dependency_rows


## 3. Export processed CSV files


In [ ]:
PROCESSED_DIR = Path('../data/processed')
if not PROCESSED_DIR.exists():
    PROCESSED_DIR = Path('../processed')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

def write_csv(path, rows, fieldnames):
    with open(path, 'w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

write_csv(PROCESSED_DIR / 'phase2_clean_hs8542_imports.csv', clean_rows,
          ['year','partner_country','trade_value_usd','reporter','flow','hs_code','commodity'])

write_csv(PROCESSED_DIR / 'taiwan_dependency_index.csv', dependency_rows,
          ['year','total_hs8542_import_usd','other_asia_nes_taiwan_proxy_import_usd',
           'taiwan_proxy_dependency_pct','total_import_yoy_growth_pct','taiwan_proxy_yoy_growth_pct'])

print('Exported processed CSV files to', PROCESSED_DIR)


## 4. Visualize trends


In [ ]:
x = [r['year'] for r in dependency_rows]
dep = [r['taiwan_proxy_dependency_pct'] for r in dependency_rows]
total_b = [r['total_hs8542_import_usd']/1e9 for r in dependency_rows]
taiwan_b = [r['other_asia_nes_taiwan_proxy_import_usd']/1e9 for r in dependency_rows]

fig = go.Figure()
fig.add_trace(go.Scatter(x=x, y=dep, mode='lines+markers', name='Other Asia, nes share (%)'))
fig.update_layout(
    title='Thailand HS8542 Import Dependency on Other Asia, nes (Taiwan Proxy), 2014–2024',
    xaxis_title='Year',
    yaxis_title='Share of total HS8542 imports (%)',
    template='plotly_white',
    hovermode='x unified'
)
fig.show()


In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=x, y=total_b, mode='lines+markers', name='Total HS8542 imports'))
fig.add_trace(go.Scatter(x=x, y=taiwan_b, mode='lines+markers', name='Other Asia, nes imports'))
fig.update_layout(
    title='Thailand HS8542 Import Value Trend, 2014–2024',
    xaxis_title='Year',
    yaxis_title='Import value (USD billions)',
    template='plotly_white',
    hovermode='x unified'
)
fig.show()
